In [5]:
import pandas as pd
from sklearn import model_selection, preprocessing
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from sklearn.metrics import mean_squared_error

In [6]:
df = pd.read_csv("customer_rating.csv")
df.head(2)

,customer_id,vendor_id,service_type,rating
0,14,14,5,1
1,13,9,1,5


In [7]:
print(f"Unique Customers: {df.customer_id.nunique()}, Unique Vendors: {df.vendor_id.nunique()}, Unique Services:{df.service_type.nunique()}")

Unique Customers: 15, Unique Vendors: 15, Unique Services:8


In [8]:
#%% Data Class (Updated)
class CustomerDataset(Dataset):
    def __init__(self, customers, vendors, ratings):
        self.customers = customers
        self.vendors = vendors
        self.ratings = ratings
    # len(movie_dataset)
    def __len__(self):
        return len(self.customers)

    def __getitem__(self, idx):
        customers = self.customers[idx]
        vendors = self.vendors[idx]
        ratings = self.ratings[idx]

        return torch.tensor(customers, dtype=torch.long), torch.tensor(vendors, dtype=torch.long),torch.tensor(ratings, dtype=torch.long),


In [9]:
class RecSysModel(nn.Module):
    def __init__(self, n_customers, n_vendors, n_embeddings = 32):
        super().__init__()
        self.customers_embed = nn.Embedding(n_customers, n_embeddings)
        self.vendors_embed = nn.Embedding(n_vendors, n_embeddings)
        self.out = nn.Linear(n_embeddings * 2, 1)

    def forward(self, customers, vendors):
        customers_embeds = self.customers_embed(customers)
        vendors_embeds = self.vendors_embed(vendors)
        x = torch.cat([customers_embeds, vendors_embeds], dim=1)
        x = self.out(x)
        return x

In [10]:
#%% encode user and movie id to start from 0 (Updated)
lbl_customer = preprocessing.LabelEncoder()
lbl_vendor = preprocessing.LabelEncoder()
df.customer_id = lbl_customer.fit_transform(df.customer_id.values)
df.vendor_id = lbl_vendor.fit_transform(df.vendor_id.values)

In [11]:
#%% create train test split
df_train, df_test = model_selection.train_test_split(
    df, test_size=0.2, random_state=42, stratify=df.rating.values
)

In [12]:
#%% Dataset Instances (Updated)
train_dataset = CustomerDataset(
    customers=df_train.customer_id.values,
    vendors=df_train.vendor_id.values,
    ratings=df_train.rating.values
)

valid_dataset = CustomerDataset(
    customers=df_test.customer_id.values,
    vendors=df_test.vendor_id.values,
    ratings=df_test.rating.values
)

In [13]:
#%% Data Loaders
BATCH_SIZE = 4
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True
                          )

test_loader = DataLoader(dataset=valid_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True
                          )

In [14]:
#%% Model Instance, Optimizer, and Loss Function
model = RecSysModel(
    n_customers=len(lbl_customer.classes_),
    n_vendors=len(lbl_vendor.classes_))

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

In [15]:
#%% Model Training
NUM_EPOCHS = 10

model.train()
for epoch_i in range(NUM_EPOCHS):
    for customers, vendors, ratings in train_loader:
        optimizer.zero_grad()
        y_pred = model(customers,
                       vendors)
        y_true = ratings.unsqueeze(dim=1).to(torch.float32)
        loss = criterion(y_pred, y_true)
        loss.backward()
        optimizer.step()


In [16]:
import itertools
#%% Model Evaluation
y_preds = []
y_trues = []

model.eval()
with torch.no_grad():
    for customers, vendors, ratings in test_loader:
        y_true = ratings.detach().numpy().tolist()
        y_pred = model(customers, vendors).squeeze().detach().numpy().tolist()

        # Handle cases where y_pred might be a single value instead of a list
        if not isinstance(y_pred, list):
            y_pred = [y_pred]
        if not isinstance(y_true, list):
            y_true = [y_true]

        y_trues.append(y_true)
        y_preds.append(y_pred)

# Flatten the lists of lists before calculating MSE
y_trues_flat = list(itertools.chain.from_iterable(y_trues))
y_preds_flat = list(itertools.chain.from_iterable(y_preds))

mse = mean_squared_error(y_trues_flat, y_preds_flat)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 2.0180086106205026


In [17]:
#%% Users and Items
customer_vendor_test = defaultdict(list)

with torch.no_grad():
    for customers, vendors, ratings in test_loader:
        y_pred = model(customers, vendors)
        for i in range(len(customers)):
            customer_id = customers[i].item()
            vendor_id = vendors[i].item()
            pred_rating = y_pred[i][0].item()
            true_rating = ratings[i].item()

            print(f"Customer: {customer_id}, Vendor: {vendor_id}, Pred: {pred_rating}, True: {true_rating}")
            customer_vendor_test[customer_id].append((pred_rating, true_rating))


Customer: 5, Vendor: 2, Pred: 3.627934694290161, True: 5
Customer: 5, Vendor: 10, Pred: 3.430514335632324, True: 1
Customer: 2, Vendor: 4, Pred: 4.182313919067383, True: 4
Customer: 1, Vendor: 5, Pred: 4.1217041015625, True: 4
Customer: 0, Vendor: 0, Pred: 3.3187906742095947, True: 5
Customer: 5, Vendor: 9, Pred: 3.7320635318756104, True: 3
Customer: 5, Vendor: 3, Pred: 4.417514324188232, True: 5
Customer: 10, Vendor: 1, Pred: 3.3540730476379395, True: 5
Customer: 7, Vendor: 1, Pred: 3.744136333465576, True: 4
Customer: 4, Vendor: 14, Pred: 3.854334831237793, True: 5
Customer: 4, Vendor: 3, Pred: 3.9353458881378174, True: 5
Customer: 7, Vendor: 2, Pred: 3.3679070472717285, True: 5
Customer: 10, Vendor: 7, Pred: 2.9111063480377197, True: 4
Customer: 1, Vendor: 7, Pred: 3.677313804626465, True: 1
Customer: 14, Vendor: 2, Pred: 3.322463274002075, True: 4
Customer: 7, Vendor: 3, Pred: 4.157486915588379, True: 5
Customer: 6, Vendor: 0, Pred: 3.904015302658081, True: 2
Customer: 0, Vendor: 1

In [18]:
#%% Precision and Recall
precisions = {}
recalls = {}

k = 10
thres = 3.5

for uid, user_ratings in customer_vendor_test.items():
    # Sort user ratings by rating
    user_ratings.sort(key=lambda x: x[0], reverse=True)

    # count of relevant items
    n_rel = sum((rating_true >= thres) for (_, rating_true) in user_ratings)

    # count recommended items that are predicted relevent and within topk
    n_rec_k = sum((rating_pred >= thres) for (rating_pred, _) in user_ratings[:k])

    # count recommended AND relevant item
    n_rel_and_rec_k = sum(
        ((rating_true >= thres) and (rating_pred >= thres))
        for (rating_pred, rating_true) in user_ratings[:k]
    )

    print(f"uid {uid},  n_rel {n_rel}, n_rec_k {n_rec_k}, n_rel_and_rec_k {n_rel_and_rec_k}")

    precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

    recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

print(f"Precision @ {k}: {sum(precisions.values()) / len(precisions)}")

print(f"Recall @ {k} : {sum(recalls.values()) / len(recalls)}")

uid 5,  n_rel 5, n_rec_k 7, n_rel_and_rec_k 5
uid 2,  n_rel 2, n_rec_k 4, n_rel_and_rec_k 2
uid 1,  n_rel 5, n_rec_k 7, n_rel_and_rec_k 5
uid 0,  n_rel 7, n_rec_k 1, n_rel_and_rec_k 1
uid 10,  n_rel 3, n_rec_k 2, n_rel_and_rec_k 1
uid 7,  n_rel 8, n_rec_k 7, n_rel_and_rec_k 6
uid 4,  n_rel 6, n_rec_k 3, n_rel_and_rec_k 3
uid 14,  n_rel 3, n_rec_k 2, n_rel_and_rec_k 1
uid 6,  n_rel 1, n_rec_k 2, n_rel_and_rec_k 1
uid 3,  n_rel 4, n_rec_k 5, n_rel_and_rec_k 4
uid 12,  n_rel 3, n_rec_k 3, n_rel_and_rec_k 3
uid 8,  n_rel 4, n_rec_k 3, n_rel_and_rec_k 2
uid 13,  n_rel 3, n_rec_k 3, n_rel_and_rec_k 3
uid 11,  n_rel 2, n_rec_k 2, n_rel_and_rec_k 2
uid 9,  n_rel 3, n_rec_k 5, n_rel_and_rec_k 3
Precision @ 10: 0.7568253968253968
Recall @ 10 : 0.7706349206349207


In [19]:
torch.save(model.state_dict(), 'recsys_model.pth')
print("Model saved as recsys_model.pth")

Model saved as recsys_model.pth


### About the Saved Model

The file `recsys_model.pth` contains the **state dictionary** of your trained `RecSysModel`.

**What is a State Dictionary?**

A state dictionary is simply a Python dictionary object that maps each layer to its learnable parameters (weights and biases). It contains everything needed to reconstruct the trained state of your model.

**Why save the state dictionary instead of the whole model?**

*   **Flexibility:** Saving only the `state_dict()` makes it more flexible. When loading, you can instantiate the model class (ensuring it has the same architecture) and then load the state dictionary into it. This is generally preferred for saving and loading models in PyTorch.
*   **Smaller File Size:** It often results in smaller file sizes compared to saving the entire model object, which might include extra information not strictly necessary for deployment or continued training.

**How to Load the Model Later:**

To load this saved model in the future, you would typically do the following:

1.  **Define the Model Architecture:** Ensure you have the `RecSysModel` class definition available.
2.  **Instantiate the Model:** Create a new instance of your `RecSysModel` with the same parameters (e.g., `n_customers`, `n_vendors`, `n_embeddings`).
3.  **Load the State Dictionary:** Load the saved state dictionary into your new model instance using `model.load_state_dict(torch.load('recsys_model.pth'))`.

For example:

```python
# Assuming RecSysModel class is defined and lbl_customer, lbl_vendor are available
loaded_model = RecSysModel(
    n_customers=len(lbl_customer.classes_),
    n_vendors=len(lbl_vendor.classes_)
)
loaded_model.load_state_dict(torch.load('recsys_model.pth'))
loaded_model.eval() # Set to evaluation mode if you're only making predictions
```

In [20]:
def get_top_n_recommendations(customer_id_original, model, lbl_customer, lbl_vendor, n=10):
    # 1. Encode the original customer_id
    try:
        customer_id_encoded = lbl_customer.transform([customer_id_original])[0]
    except ValueError:
        print(f"Customer ID {customer_id_original} not found in training data.")
        return []

    # 2. Get all unique encoded vendor IDs
    all_vendor_ids_encoded = torch.arange(len(lbl_vendor.classes_))

    # 3. Create tensors for prediction
    customer_tensor = torch.tensor([customer_id_encoded] * len(all_vendor_ids_encoded), dtype=torch.long)
    vendor_tensor = all_vendor_ids_encoded

    # 4. Get predictions from the model
    model.eval()
    with torch.no_grad():
        predictions = model(customer_tensor, vendor_tensor).squeeze().tolist()

    # 5. Map predictions back to original vendor IDs
    # Create a list of (predicted_rating, encoded_vendor_id)
    vendor_ratings = []
    for i, pred_rating in enumerate(predictions):
        vendor_id_encoded = all_vendor_ids_encoded[i].item()
        vendor_id_original = lbl_vendor.inverse_transform([vendor_id_encoded])[0]
        vendor_ratings.append((pred_rating, vendor_id_original))

    # 6. Sort by predicted rating in descending order
    vendor_ratings.sort(key=lambda x: x[0], reverse=True)

    # 7. Return the top N vendors
    return vendor_ratings[:n]

# Example Usage:
# Let's pick an original customer_id that exists in our dataset
# You can pick any customer_id from df.customer_id.unique() before label encoding
# For simplicity, let's use one of the original customer IDs from the raw data

# To demonstrate, let's get an original customer ID from the non-encoded dataframe
# (assuming you have access to the original df before encoding)
# If not, you might need to inverse transform an encoded ID.

# Let's assume you want recommendations for original customer ID 14 (which is encoded to 13)
original_customer_id_to_recommend = 14 # Example: use an original customer ID from your CSV

top_vendors = get_top_n_recommendations(original_customer_id_to_recommend, model, lbl_customer, lbl_vendor, n=10)

print(f"Top 10 recommended vendors for Customer ID {original_customer_id_to_recommend}:")
for rating, vendor_id in top_vendors:
    print(f"  Vendor ID: {vendor_id}, Predicted Rating: {rating:.2f}")

Top 10 recommended vendors for Customer ID 14:
  Vendor ID: 7, Predicted Rating: 4.08
  Vendor ID: 4, Predicted Rating: 3.93
  Vendor ID: 15, Predicted Rating: 3.85
  Vendor ID: 5, Predicted Rating: 3.73
  Vendor ID: 12, Predicted Rating: 3.68
  Vendor ID: 13, Predicted Rating: 3.65
  Vendor ID: 6, Predicted Rating: 3.51
  Vendor ID: 2, Predicted Rating: 3.51
  Vendor ID: 14, Predicted Rating: 3.49
  Vendor ID: 1, Predicted Rating: 3.35
